# Generate `threshold.json` for META-CXR

Notebook này tạo `threshold.json` mới từ **validation set** của dataset đang dùng. Không tune trên test set.

Nguồn dữ liệu/checkpoint:
- Checkpoint lấy từ Google Cloud Storage bucket `gs://meta-cxr-checkpoint`.
- `mimic-cxr-jpg-lite`: chứa ảnh + CheXpert CSV. Hỗ trợ mount trực tiếp hoặc dưới `/kaggle/input/datasets/phuong20052`.
- `mimic-cxr-p10-processed`: chứa `train.csv`, `val.csv`, `test.csv`. Hỗ trợ mount trực tiếp hoặc dưới `/kaggle/input/datasets/phuong20052`.
- Source/dataset chứa code `META-CXR` nếu notebook không nằm sẵn trong repo.

Nếu bucket private, thêm Kaggle Secret `GCP_SERVICE_ACCOUNT_JSON` hoặc `GCS_SERVICE_ACCOUNT` chứa nguyên nội dung service-account JSON có quyền đọc bucket.

Output:
- `/kaggle/working/threshold.json`
- `/kaggle/working/threshold_search_details.csv`

In [ ]:
# Kaggle: bật Internet nếu môi trường chưa có đủ dependency/cache model.
# Không cài lại numpy/pandas ở Kaggle Python 3.12 để tránh binary incompatibility.
# Không pin transformers==4.30.2 ở Kaggle mới vì nó kéo tokenizers<0.14 và có thể phải build từ source.
# Code Qformer hiện đã fallback sang transformers.pytorch_utils nên dùng transformers sẵn có của Kaggle là ổn.
!pip install -q \
  "omegaconf==2.3.0" \
  iopath timm scikit-image accelerate sentencepiece protobuf \
  iterative-stratification einops fairscale pycocoevalcap webdataset decord \
  ftfy regex hi-ml-multimodal torchinfo google-cloud-storage

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/minhphuong150505/Meta-CXR-Kaggle.git"
REPO_DIR = Path("/kaggle/working/META-CXR")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already exists at {REPO_DIR}; using existing checkout.")

# Change working directory to repo root
os.chdir(str(REPO_DIR))
print(f"Working directory: {os.getcwd()}")
!ls -la

In [ ]:
import os
import sys
import shutil
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')
OWNER_DATASET_ROOT = INPUT_DIR / 'datasets' / 'phuong20052'

def is_meta_cxr_project(path: Path) -> bool:
    return (path / 'model' / 'lavis').exists() and (path / 'pretraining').exists()

project_candidates = [
    Path.cwd(),
    WORK_DIR / 'META-CXR',
    WORK_DIR / 'Meta-CXR-Kaggle',
    WORK_DIR / 'Meta-CXR-Kaggle' / 'META-CXR',
]
if INPUT_DIR.exists():
    for root in INPUT_DIR.glob('*'):
        project_candidates.extend([root, root / 'META-CXR'])

source_project = next((p for p in project_candidates if is_meta_cxr_project(p)), None)
if source_project is None:
    raise FileNotFoundError('Không tìm thấy code META-CXR. Hãy attach dataset/source chứa thư mục META-CXR.')

PROJECT_DIR = WORK_DIR / 'META-CXR'
if source_project.resolve() != PROJECT_DIR.resolve():
    if PROJECT_DIR.exists() and is_meta_cxr_project(PROJECT_DIR):
        pass
    else:
        ignore = shutil.ignore_patterns('.git', 'wandb', '__pycache__', '*.pyc', 'output', 'outputs', 'checkpoints')
        shutil.copytree(source_project, PROJECT_DIR, dirs_exist_ok=True, ignore=ignore)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'model'))

def _candidate_roots(base: Path):
    yield base
    versions = base / 'versions'
    if versions.exists():
        for version_dir in sorted(versions.iterdir()):
            if version_dir.is_dir():
                yield version_dir

def find_dataset_root(name_candidates, required_files):
    search_bases = []
    for name in name_candidates:
        search_bases.extend([
            OWNER_DATASET_ROOT / name,
            INPUT_DIR / 'datasets' / name,
            INPUT_DIR / name,
        ])

    # Fallback for Kaggle layouts where title/case differs from slug.
    for root in [OWNER_DATASET_ROOT, INPUT_DIR / 'datasets', INPUT_DIR]:
        if root.exists():
            for child in sorted(root.iterdir()):
                if child.is_dir():
                    search_bases.append(child)

    checked = []
    seen = set()
    for base in search_bases:
        for candidate in _candidate_roots(Path(base)):
            key = str(candidate)
            if key in seen:
                continue
            seen.add(key)
            checked.append(key)
            if candidate.exists() and all((candidate / rel).exists() for rel in required_files):
                return candidate

    raise FileNotFoundError(
        'Không tìm thấy dataset có đủ files ' + str(required_files) + '\nĐã kiểm tra:\n' + '\n'.join(checked[:80])
    )

IMAGE_ROOT = find_dataset_root(
    ['mimic-cxr-jpg-lite', 'MIMIC-CXR-JPG-LITE'],
    ['mimic-cxr-2.0.0-split.csv', 'mimic-cxr-2.0.0-chexpert.csv', 'mimic-cxr-2.0.0-metadata.csv'],
)
PROCESSED_ROOT = find_dataset_root(
    ['mimic-cxr-p10-processed', 'mimic-cxr-p10-preprocessed', 'MIMIC-CXR p10 Preprocessed'],
    ['train.csv', 'val.csv', 'test.csv'],
)
GCS_BUCKET = 'meta-cxr-checkpoint'
CHECKPOINT_ROOT = WORK_DIR / 'checkpoints'
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

(PROJECT_DIR / 'configs').mkdir(exist_ok=True)
(PROJECT_DIR / 'configs' / 'env_config.yaml').write_text(f'''paths:
  data_root: "{IMAGE_ROOT}"
  mimic_cxr_jpg_root: "{IMAGE_ROOT}"
  split_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-split.csv"
  reports_csv: "/kaggle/working/mimic_cxr_cleaned.csv"
  chexpert_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-chexpert.csv"
  metadata_csv: "{IMAGE_ROOT}/mimic-cxr-2.0.0-metadata.csv"
  processed_dir: "{PROCESSED_ROOT}"
  processed_train_csv: "{PROCESSED_ROOT}/train.csv"
  processed_val_csv: "{PROCESSED_ROOT}/val.csv"
  processed_test_csv: "{PROCESSED_ROOT}/test.csv"
  output_dir: "/kaggle/working/output"
  checkpoint_dir: "{CHECKPOINT_ROOT}"
wandb:
  entity: ""
  project: "meta-cxr-encoder-comparison"
java:
  home: "/usr/lib/jvm/java-11-openjdk-amd64"
  path: "/usr/lib/jvm/java-11-openjdk-amd64/bin:"
''')

print('PROJECT_DIR    =', PROJECT_DIR)
print('IMAGE_ROOT     =', IMAGE_ROOT)
print('PROCESSED_ROOT =', PROCESSED_ROOT)
print('GCS_BUCKET     = gs://' + GCS_BUCKET)
print('CHECKPOINT_ROOT=', CHECKPOINT_ROOT)

# The threshold notebook does not use pandas/cv2 utilities, but mhcac.utils imports
# them at module import time. On Kaggle Python 3.12 this can break after NumPy changes.
# Patch the writable cloned repo so optional utilities are lazy/non-fatal.
utils_path = PROJECT_DIR / 'mhcac' / 'utils.py'
utils_text = utils_path.read_text(encoding='utf-8')
# Remove any pandas import line, including broken indented leftovers from an earlier patch.
utils_text = ''.join(
    line for line in utils_text.splitlines(keepends=True)
    if line.strip() != 'import pandas as pd'
)
good_cv2_patch = "try:\n    import cv2\nexcept Exception:\n    cv2 = None\n"
bad_cv2_patches = [
    "try:\n    import cv2\nexcept ImportError:\n    cv2 = None\n",
    "try:\nimport cv2\nexcept Exception:\n    cv2 = None\n",
    "try:\ntry:\n    import cv2\nexcept Exception:\n    cv2 = None\nexcept ImportError:\n    cv2 = None\n",
    "try:\n    try:\n    import cv2\nexcept Exception:\n    cv2 = None\nexcept ImportError:\n    cv2 = None\n",
]
for bad_cv2_patch in bad_cv2_patches:
    utils_text = utils_text.replace(bad_cv2_patch, good_cv2_patch)
if good_cv2_patch not in utils_text:
    utils_text = utils_text.replace('\nimport cv2\n', '\n' + good_cv2_patch, 1)
save_csv_pandas_block = '    import pandas as pd\n    df = pd.DataFrame(combined_results)'
if '    df = pd.DataFrame(combined_results)' in utils_text and save_csv_pandas_block not in utils_text:
    utils_text = utils_text.replace(
        '    df = pd.DataFrame(combined_results)',
        save_csv_pandas_block,
        1,
    )
compile(utils_text, str(utils_path), 'exec')
utils_path.write_text(utils_text, encoding='utf-8')
print('Patched mhcac/utils.py optional pandas/cv2 imports for this Kaggle session.')

# Patch Qformer for newer transformers versions where pruning/chunking helpers moved
# or were removed from modeling_utils/pytorch_utils.
qformer_path = PROJECT_DIR / 'model' / 'lavis' / 'models' / 'blip2_models' / 'Qformer.py'
qformer_text = qformer_path.read_text(encoding='utf-8')
old_import_block = """from transformers.modeling_utils import PreTrainedModel
try:
    from transformers.modeling_utils import (
        apply_chunking_to_forward,
        find_pruneable_heads_and_indices,
        prune_linear_layer,
    )
except ImportError:
    from transformers.pytorch_utils import (
        apply_chunking_to_forward,
        find_pruneable_heads_and_indices,
        prune_linear_layer,
    )
"""
new_import_block = """from transformers.modeling_utils import PreTrainedModel
try:
    from transformers.pytorch_utils import apply_chunking_to_forward
except ImportError:
    try:
        from transformers.modeling_utils import apply_chunking_to_forward
    except ImportError:
        def apply_chunking_to_forward(forward_fn, chunk_size, chunk_dim, *input_tensors):
            if len(input_tensors) == 0:
                raise ValueError('input_tensors must not be empty')
            tensor_shape = input_tensors[0].shape[chunk_dim]
            if any(t.shape[chunk_dim] != tensor_shape for t in input_tensors):
                raise ValueError('All input tensors must have the same shape on chunk_dim')
            if chunk_size > 0:
                if tensor_shape % chunk_size != 0:
                    raise ValueError('The dimension to be chunked must be a multiple of chunk_size')
                num_chunks = tensor_shape // chunk_size
                input_chunks = tuple(t.chunk(num_chunks, dim=chunk_dim) for t in input_tensors)
                output_chunks = tuple(forward_fn(*chunk) for chunk in zip(*input_chunks))
                return torch.cat(output_chunks, dim=chunk_dim)
            return forward_fn(*input_tensors)

try:
    from transformers.pytorch_utils import find_pruneable_heads_and_indices, prune_linear_layer
except ImportError:
    try:
        from transformers.modeling_utils import find_pruneable_heads_and_indices, prune_linear_layer
    except ImportError:
        def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
            heads = set(heads) - already_pruned_heads
            mask = torch.ones(n_heads, head_size)
            heads = sorted(heads)
            for head in heads:
                head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
                mask[head] = 0
            mask = mask.view(-1).contiguous().eq(1)
            index = torch.arange(len(mask))[mask].long()
            return heads, index

        def prune_linear_layer(layer, index, dim=0):
            index = index.to(layer.weight.device)
            W = layer.weight.index_select(dim, index).clone().detach()
            if layer.bias is not None:
                b = layer.bias.clone().detach() if dim == 1 else layer.bias[index].clone().detach()
            new_size = list(layer.weight.size())
            new_size[dim] = len(index)
            new_layer = nn.Linear(new_size[1], new_size[0], bias=layer.bias is not None).to(layer.weight.device)
            new_layer.weight.requires_grad = False
            new_layer.weight.copy_(W.contiguous())
            new_layer.weight.requires_grad = True
            if layer.bias is not None:
                new_layer.bias.requires_grad = False
                new_layer.bias.copy_(b.contiguous())
                new_layer.bias.requires_grad = True
            return new_layer
"""
if old_import_block in qformer_text:
    qformer_text = qformer_text.replace(old_import_block, new_import_block)
    qformer_path.write_text(qformer_text, encoding='utf-8')
    print('Patched Qformer.py transformers helper imports for this Kaggle session.')
else:
    print('Qformer.py import block already patched or differs from expected pattern.')

qformer_text = qformer_path.read_text(encoding='utf-8')
if 'all_tied_weights_keys = {}' not in qformer_text:
    qformer_text = qformer_text.replace(
        'class BertPreTrainedModel(PreTrainedModel):\n',
        'class BertPreTrainedModel(PreTrainedModel):\n    all_tied_weights_keys = {}\n',
    )
    qformer_path.write_text(qformer_text, encoding='utf-8')
    print('Patched Qformer.py all_tied_weights_keys for newer transformers.')

# If a previous cell failed during import, remove partial modules before retrying.
for module_name in list(sys.modules):
    if module_name.startswith('model.lavis') or module_name.startswith('mhcac'):
        sys.modules.pop(module_name, None)

## Chọn checkpoint dùng để tune threshold

Thông thường nên dùng checkpoint của model cuối cùng bạn dùng để inference/report generation. Nếu bạn muốn tune threshold cho từng encoder riêng, chạy notebook này nhiều lần và đổi `RUN_NAME`.

`find_checkpoint(RUN_NAME)` sẽ ưu tiên file đã có trong `/kaggle/working/checkpoints`; nếu chưa có, nó tự tải `checkpoint_best.pth` từ `gs://meta-cxr-checkpoint`, fallback sang `checkpoint_last.pth`.

In [ ]:
# Đổi RUN_NAME nếu muốn tạo threshold cho encoder khác.
RUN_NAME = '07_all_three'

# Dùng validation set để tune threshold. Không đổi sang test nếu dùng cho báo cáo kết quả.
SPLIT = 'val'

# Lưới threshold. 0.01 đủ mịn cho validation nhỏ; có thể giảm step nếu cần.
THRESHOLD_GRID = [round(x / 100, 2) for x in range(1, 100)]

EVAL_BATCH_SIZE = 4
NUM_WORKERS = 2

In [ ]:
import csv
import gc
import json
from types import SimpleNamespace

import numpy as np
import torch
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import model.lavis.tasks as tasks
from model.lavis.common.config import Config
from model.lavis.common.registry import registry

# Registration imports.
from model.lavis.common.optims import LinearWarmupCosineLRScheduler, LinearWarmupStepLRScheduler
from model.lavis.datasets.builders import *
from model.lavis.models import *
from model.lavis.processors import *
from model.lavis.tasks import *
from model.lavis.data.ReportDataset import MIMIC_CXR_Dataset
from local_config import VIS_ROOT

registry.mapping['paths']['cache_root'] = '.'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CHEXPERT_COLS = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis',
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
]

# Index class trong code hiện tại: negative=0, positive=1, uncertain=2.
# No Finding và Support Devices có thể chỉ có 2 class trong threshold cũ, nhưng model vẫn xuất 3 class.
CLASS_MAP = {'negative': 0, 'positive': 1, 'uncertain': 2}

print('DEVICE =', DEVICE)

In [ ]:
def _local_checkpoint_candidates(run_name: str, filename: str) -> list[Path]:
    return [p for p in CHECKPOINT_ROOT.rglob(filename) if run_name in str(p)]

def _get_gcs_client():
    from google.cloud import storage

    secret_value = None
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for secret_name in ('GCP_SERVICE_ACCOUNT_JSON', 'GCS_SERVICE_ACCOUNT'):
            try:
                secret_value = secrets.get_secret(secret_name)
            except Exception:
                secret_value = None
            if secret_value:
                print(f'Using Kaggle Secret {secret_name} for GCS auth.')
                break
    except Exception:
        secret_value = None

    if secret_value:
        import json as _json
        from google.oauth2 import service_account

        sa_info = _json.loads(secret_value)
        credentials = service_account.Credentials.from_service_account_info(sa_info)
        return storage.Client(project=sa_info.get('project_id'), credentials=credentials)

    try:
        return storage.Client()
    except Exception:
        print('No GCS credentials found; trying anonymous client. Private buckets will fail.')
        return storage.Client.create_anonymous_client()

def _download_checkpoint_from_gcs(run_name: str) -> Path:
    client = _get_gcs_client()
    all_blobs_loaded = False
    try:
        blobs = list(client.list_blobs(GCS_BUCKET, prefix=f'{run_name}/'))
        if not blobs:
            blobs = list(client.list_blobs(GCS_BUCKET))
            all_blobs_loaded = True
    except Exception as exc:
        raise RuntimeError(
            f'Không đọc được gs://{GCS_BUCKET}. Bật Internet trong Kaggle và set '
            'Kaggle Secret GCP_SERVICE_ACCOUNT_JSON hoặc GCS_SERVICE_ACCOUNT nếu bucket private.'
        ) from exc

    candidates = [
        b for b in blobs
        if run_name in b.name and b.name.endswith(('checkpoint_best.pth', 'checkpoint_last.pth'))
    ]
    if not candidates and not all_blobs_loaded:
        blobs = list(client.list_blobs(GCS_BUCKET))
        candidates = [
            b for b in blobs
            if run_name in b.name and b.name.endswith(('checkpoint_best.pth', 'checkpoint_last.pth'))
        ]

    if not candidates:
        seen = '\n'.join(f'  - {b.name}' for b in blobs[:30])
        raise FileNotFoundError(
            f'Không có checkpoint_best/last cho {run_name} trong gs://{GCS_BUCKET}. '
            f'Các blob thấy được:\n{seen}'
        )

    best = [b for b in candidates if b.name.endswith('checkpoint_best.pth')]
    chosen = sorted(best or candidates, key=lambda b: len(b.name))[0]
    local_path = CHECKPOINT_ROOT / chosen.name
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists():
        print(f'Using cached checkpoint {local_path}')
        return local_path

    print(f'Downloading gs://{GCS_BUCKET}/{chosen.name} -> {local_path}')
    chosen.download_to_filename(str(local_path))
    print(f'OK ({local_path.stat().st_size / 1e6:.1f} MB)')
    return local_path

def find_checkpoint(run_name: str) -> Path:
    best = _local_checkpoint_candidates(run_name, 'checkpoint_best.pth')
    if best:
        return sorted(best, key=lambda p: len(str(p)))[0]
    last = _local_checkpoint_candidates(run_name, 'checkpoint_last.pth')
    if last:
        print(f'WARNING: {run_name}: checkpoint_best.pth not found locally, using checkpoint_last.pth')
        return sorted(last, key=lambda p: len(str(p)))[0]
    return _download_checkpoint_from_gcs(run_name)

def build_cfg(run_name: str):
    cfg_path = PROJECT_DIR / 'pretraining' / 'configs' / 'encoder_comparison' / f'{run_name}.yaml'
    args = SimpleNamespace(cfg_path=str(cfg_path), options=None)
    return Config(args)

def build_model_for_run(run_name: str, checkpoint_path: Path):
    cfg = build_cfg(run_name)
    task = tasks.setup_task(cfg)
    model = task.build_model(cfg)
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    state_dict = ckpt['model'] if isinstance(ckpt, dict) and 'model' in ckpt else ckpt
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f'{run_name}: loaded {checkpoint_path.name}; missing={len(missing)}, unexpected={len(unexpected)}')
    model.to(DEVICE)
    model.eval()
    return cfg, model

def make_loader(cfg, split: str):
    dataset = MIMIC_CXR_Dataset(
        vis_processor=None,
        text_processor=None,
        vis_root=VIS_ROOT,
        split=split,
        cfg=cfg,
        truncate=None,
    )
    return DataLoader(
        dataset,
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
    )

@torch.no_grad()
def predict_logits_with_text(model, batch):
    image = batch['image'].to(DEVICE, non_blocking=True)
    text = batch['text_output']

    cnn_patches, vit_patches, swin_patches, _ = model._encode_image_streams(image, apply_aug=False)
    text_tokens = model.tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=model.max_txt_len,
        return_tensors='pt',
    ).to(DEVICE)
    text_output = model.Qformer.bert(
        text_tokens.input_ids,
        attention_mask=text_tokens.attention_mask,
        return_dict=True,
    )
    logits, _, _, _, _ = model.mhcac(
        cnn_patches=cnn_patches,
        vit_patches=vit_patches,
        swin_patches=swin_patches,
        text_embeddings=text_output.last_hidden_state,
        labels=None,
    )
    return logits

In [ ]:
checkpoint_path = find_checkpoint(RUN_NAME)
cfg, model = build_model_for_run(RUN_NAME, checkpoint_path)
loader = make_loader(cfg, SPLIT)

all_probs = []
all_labels = []

for batch in tqdm(loader, desc=f'{RUN_NAME}:{SPLIT}'):
    logits = predict_logits_with_text(model, batch)
    probs = torch.softmax(logits, dim=-1)
    all_probs.append(probs.cpu().numpy())
    all_labels.append(batch['classification_labels'].cpu().numpy())

probs = np.concatenate(all_probs, axis=0)      # [N, 14, 3]
labels = np.concatenate(all_labels, axis=0)    # [N, 14]

print('probs shape =', probs.shape)
print('labels shape =', labels.shape)

del model, loader
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

In [ ]:
def best_threshold_for_binary(y_true_binary, scores, grid):
    best_t = 0.5
    best_f1 = -1.0
    for t in grid:
        y_pred_binary = (scores >= t).astype(int)
        score = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        if score > best_f1:
            best_f1 = float(score)
            best_t = float(t)
    return best_t, best_f1

thresholds = {}
detail_rows = []

for task_idx, abnormality in enumerate(CHEXPERT_COLS):
    thresholds[abnormality] = {}
    y_task = labels[:, task_idx]
    for class_name, class_idx in CLASS_MAP.items():
        # Nếu validation set không có class này cho abnormality đó thì bỏ qua.
        y_true_binary = (y_task == class_idx).astype(int)
        positives = int(y_true_binary.sum())
        if positives == 0:
            continue
        scores = probs[:, task_idx, class_idx]
        best_t, best_f1 = best_threshold_for_binary(y_true_binary, scores, THRESHOLD_GRID)
        thresholds[abnormality][class_name] = best_t
        detail_rows.append({
            'abnormality': abnormality,
            'class': class_name,
            'threshold': best_t,
            'val_f1': best_f1,
            'positive_count': positives,
            'n': len(y_true_binary),
        })

threshold_path = Path('/kaggle/working/threshold.json')
details_path = Path('/kaggle/working/threshold_search_details.csv')

threshold_path.write_text(json.dumps(thresholds, indent=4), encoding='utf-8')
detail_rows_sorted = sorted(detail_rows, key=lambda row: (row['abnormality'], row['class']))
with details_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=['abnormality', 'class', 'threshold', 'val_f1', 'positive_count', 'n'],
    )
    writer.writeheader()
    writer.writerows(detail_rows_sorted)

print('Saved', threshold_path)
print('Saved', details_path)
for row in detail_rows_sorted[:20]:
    print(row)
print(json.dumps(thresholds, indent=4)[:2000])

## Ghi chú sử dụng

- File `/kaggle/working/threshold.json` sinh từ validation set, phù hợp hơn với subset/dataset hiện tại so với threshold cũ từ dataset lớn hơn.
- Không dùng file này để báo cáo test F1 nếu bạn đã tune trực tiếp trên test.
- Nếu inference dùng `07_all_three`, nên tạo threshold với `RUN_NAME = '07_all_three'`.
- Nếu muốn mỗi encoder có threshold riêng, chạy lại notebook cho từng `RUN_NAME` và lưu tên file riêng, ví dụ `threshold_01_biovil_only.json`.